In [1]:
import json
import copy
from Bio import SeqIO

import pandas as pd
import numpy as np

In [2]:
df = pd.read_table("../data/NV2g.20240221.geneNames.txt", index_col="id")
df = df.dropna(subset=["name"])
df.head()

,main_id,name,sym,source_id,oma_group_id
id,,,,,
NV2t009957001.1,SELF,NV2t009957001.1,NaN,0,0
NV2t005400001.1,SELF,aurora kinase A,AURKA,OMA|OrthologousGroup|HUMAN|ENSG00000087586,OMAGROUP|OGNone
NV2t011205001.1,SELF,KAT8 regulatory NSL complex subunit 1,KANSL1,OMA|1to1|ENS|HUMAN|ENSG00000120071,OMAGROUP|OG1052381
NV2t014228001.1,SELF,Filamin-A,Flna,UNIPROT|Mus musculus|Q8BTM8|6.66e-07,0
NV2t006138001.1,SELF,Derived by automated computational analysis us...,0,OMA|1to1|MultiGroup|MULTI|OG845447(HAECO17377)...,"OMAGROUP|OGMULTI|OG845447(HAECO17377),OG845447..."


In [3]:
df.loc["NV2t014228001.1"]["name"]

'Filamin-A'

In [4]:
df.shape

(35568, 5)

In [5]:
prot_seqs = {}
ids = []
with open("../data/proteins_v2.fa") as handle:
    for record in SeqIO.parse(handle, "fasta"):
        ids.append(record.id)
        prot_seqs[record.id] = record.seq

In [6]:
base_json_dict = {
    "name": "",
    "modelSeeds": [],
    "sequences": [
        {
            "proteinChain": {
                "sequence": "SPKPPGLW",
                "count": 1,
                "useStructureTemplate": True
            }
        },
        {
            "proteinChain": {
                "sequence": "",
                "count": 1,
                "useStructureTemplate": True
                }
            }
    ],
    "dialect": "alphafoldserver",
    "version": 3
}

In [7]:
current_list = []
num_jsons = 1
for i in range(len(ids)):
    current_id = ids[i]
    try:
        current_name = df.loc[current_id]["name"]
    except:
        continue
    #print(current_name)
    if ("G-protein coupled receptor" in current_name) or ("ion channel" in current_name):
        current_seq = str(prot_seqs[current_id])
        current_dict = copy.deepcopy(base_json_dict)
        current_dict["name"] = current_id[:-2] + "_GLW6"
        current_dict["sequences"][1]["proteinChain"]["sequence"] = current_seq
        current_list.append(current_dict)
    if len(current_list) == 30 or i == len(ids) - 1:
        filename = "output_GLW6_" + str(num_jsons) + ".json"
        with open(filename, "w") as fw:
            json.dump(current_list, fw, indent=4)
        num_jsons += 1
        current_list = []

Below: Old work that is now outdated (using old, unprocessed sequences); some notes from there are still helpful

In [2]:
prot_seqs = {}
prot_descs = {}
ids = []
with open("../data/proteins.fasta") as handle:
    for record in SeqIO.parse(handle, "fasta"):
        ids.append(record.id)
        prot_seqs[record.id] = record.seq
        prot_descs[record.id] = record.description

In [ ]:
#come back to this; we may need to use the shortest portion of GLW\
#do we know the GLW version that works?
base_json_dict = {
    "name": "",
    "modelSeeds": [],
    "sequences": [
        {
            "proteinChain": {
                "sequence": "LFGHTLVAVLFCLALCHAETKRKAADTTDTENELDASPNVNNDDDNDIKRMPTDTKRQAGAPGLWGKRDAGPPGLWGKRDAGPPGLWGKRSPKPPGLWGKRQAGAPGLWGKRSAGPPGLWGKRDAGPPGLWGKRVAGPPGLWGKRQAGAPGLWGREAGAPGLWGKRQAGAPGLWGKREAGAPGLWGREANAPGLWGKRRAGAPGLWGKREANAPGLWGKRQAGPPGLWGKREANAPGLWGKRQAGPPGLWGKREANAPGLWGKRQAGPPGLWGKRDEDEDEDMDETNGDPLWGRSADAGPPGLWGRKKRAASPQRDLYGIGLWGRNAALMTAEELDLSFKNEEQS",
                "count": 1,
                "useStructureTemplate": True
            }
        },
        {
            "proteinChain": {
                "sequence": "",
                "count": 1,
                "useStructureTemplate": True
                }
            }
    ],
    "dialect": "alphafoldserver",
    "version": 3
}

In [ ]:
#there are potentially some GPCRs that we could focus on specifically. 
#can look at the Nakanishi and Martindale paper
#as well as cells that are positive for ETS
#look at ion channels too
#could look into version 2??? would need to consult a different database, like simrbase

current_list = []
num_jsons = 1
for i in range(len(ids)):
    current_id = ids[i]
    current_desc = prot_descs[current_id]
    if "G-protein coupled receptor" in current_desc:
        current_seq = str(prot_seqs[current_id])
        current_dict = copy.deepcopy(base_json_dict)
        current_dict["name"] = current_id[3:9]
        current_dict["sequences"][1]["proteinChain"]["sequence"] = current_seq
        current_list.append(current_dict)
    if len(current_list) == 30 or i == len(ids) - 1:
        filename = "output_" + str(num_jsons) + ".json"
        with open(filename, "w") as fw:
            json.dump(current_list, fw, indent=4)
        num_jsons += 1
        current_list = []
    